# Experiments

### Setup

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv()

True

Here is the RAG Application that we've been working with throughout this course

In [2]:
import os
import nest_asyncio
from typing import List
from dotenv import load_dotenv
from openai import OpenAI
from langsmith import traceable
from app import get_vector_db_retriever

load_dotenv()

# --- Stack Configuration ---
MODEL_NAME = "openai/gpt-oss-120b"  # Experiment 1 Model
MODEL_PROVIDER = "groq"
APP_VERSION = 1.0

RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

nest_asyncio.apply()
retriever = get_vector_db_retriever()


@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)


@traceable(run_type="chain")
def generate_response(question: str, documents, model: str = MODEL_NAME):
    # Rule #5: Token Diet
    formatted_docs = "\n\n".join(doc.page_content[:600] for doc in documents)
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Context: {formatted_docs} \n\n Question: {question}"}
    ]
    return call_openai(messages, model=model)


@traceable(run_type="llm")
def call_openai(messages: List[dict], model: str = MODEL_NAME) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=150
    )


@traceable(run_type="chain")
def langsmith_rag(question: str, model: str = MODEL_NAME):
    documents = retrieve_documents(question)
    response = generate_response(question, documents, model=model)
    return response.choices[0].message.content

### Experiment

Here is a code snippet that should look similar to what you see from the starter code!

There are a few important components here.

1. We have defined an Evaluator
2. We pipe our dataset examples (dict) to the shape of input that our function `langsmith_rag` takes (str) using a target function

In [3]:
from langsmith import evaluate, Client

client = Client()

# NOTE: Use the exact name of the dataset you created in Lesson 1
dataset_name = "LangSmith RAG Q&A Dataset" 

def is_concise_enough(reference_outputs: dict, outputs: dict) -> dict:
    candidate = outputs.get("output") if outputs else None
    reference = reference_outputs.get("output") if reference_outputs else ""
    
    # If the model failed or returned None, score it 0 instead of crashing
    if not candidate:
        return {"key": "is_concise", "score": 0}

    score = len(candidate) < 1.5 * len(reference)
    return {"key": "is_concise", "score": int(score)}

def target_function(inputs: dict):
    return langsmith_rag(inputs["question"], model="openai/gpt-oss-120b")

# Run Experiment 1
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="groq-gpt-oss-120b"
)

View the evaluation results for experiment: 'groq-gpt-oss-120b-570a6919' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=9d107241-d724-48a1-b1cb-d08bb0736b39




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,Use the `metadata` argument of the `@traceable...,None,You can pass metadata with the @traceable deco...,1,1.193479,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a11cb2-17d8-7383-a4a9-fca64947cfb0
1,Does LangSmith support offline evaluation?,Yes. LangSmith provides offline evaluation for...,None,"Yes, LangSmith supports offline evaluation thr...",1,0.695072,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a11cb2-1c83-70b0-8839-812399151a03
2,Can LangSmith be used for finetuning and model...,LangSmith is a monitoring and evaluation platf...,None,"Yes, LangSmith can be used for fine-tuning and...",1,1.487947,366693f8-2a71-43fc-9118-3d0427fb6c79,01a11cb2-1f3c-7b20-b721-af5f98dc1ca3
3,Can LangSmith be used to evaluate agents?,Yes. LangSmith’s evaluation framework works wi...,None,"Yes, LangSmith can be used to evaluate agents....",1,0.777840,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a11cb2-250d-7143-bd89-0644b133684a
4,How do I create user feedback with the LangSmi...,Use the LangSmith Python SDK’s feedback endpoint.,None,To create user feedback with the LangSmith SDK...,1,1.107788,8401e038-1738-4410-9b1a-89dc755696ea,01a11cb2-2818-7ad2-85dd-36046cefcca3
5,Is there a JavaScript LangSmith SDK?,Yes—LangSmith provides a JavaScript/TypeScript...,None,"Yes, there is a JavaScript LangSmith SDK!",0,0.885345,eb396e7b-683b-4a6a-a085-86ed959ded80,01a11cb2-2c6d-7fb2-89dc-b1e9e718cdbf
6,How do I set up tracing to LangSmith if I'm us...,"To enable LangSmith tracing with LangChain, se...",None,Set your LangChain app to use LangSmith’s trac...,1,0.837246,c7fb22d1-c4c5-49dd-92c3-da90a804bc97,01a11cb2-2fe3-7570-a6cf-f59702c63f27
7,How can I trace with the @traceable decorator?,Apply the @traceable decorator to any Python f...,None,To trace with the @traceable decorator in Pyth...,1,1.301942,0e3e0494-ccd1-4a20-b605-be94e398fee5,01a11cb2-332a-78c1-89b3-eeea3b073ac3
8,What is LangSmith used for in three sentences?,LangSmith is a platform for tracing and monito...,None,LangSmith is a platform designed for the devel...,1,0.681026,34bf2ff8-0797-4828-a770-cb6289e29049,01a11cb2-3841-72c3-84fc-94b6a8ba43fa
9,What testing capabilities does LangSmith have?,LangSmith includes built‑in evaluation tools t...,None,LangSmith offers capabilities for creating dat...,1,0.592862,604440fa-b554-46d5-aad2-2a5e41d3759d,01a11cb2-3b06-7301-bac6-c65a141cce6b


### Modifying your Application

Now, let's change our model to gpt-35-turbo and see how it performs!

Make this change, and then run this code snippet!

In [5]:
def target_function_qwen(inputs: dict):
    return langsmith_rag(inputs["question"], model="qwen/qwen3.8-27b")

evaluate(
    target_function_qwen,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="groq-qwen-27b"
)

View the evaluation results for experiment: 'groq-qwen-27b-811f7cbc' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=f46d3e98-8aad-4f7e-a70c-691454f4e909




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,You can pass metadata to the `@traceable` deco...,None,You can pass metadata with the @traceable deco...,1,0.565262,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a11cb2-849f-7ac0-9685-d62f3d815e6a
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation, wh...",None,"Yes, LangSmith supports offline evaluation thr...",1,0.420372,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a11cb2-86d6-7572-93fa-ab1c1f425038
2,Can LangSmith be used for finetuning and model...,"No, the provided context indicates that LangSm...",None,"Yes, LangSmith can be used for fine-tuning and...",1,0.424727,366693f8-2a71-43fc-9118-3d0427fb6c79,01a11cb2-887b-7d32-954b-03e5258d4349
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents,...",None,"Yes, LangSmith can be used to evaluate agents....",1,0.435234,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a11cb2-8a25-73e1-8f45-08929e60989f
4,How do I create user feedback with the LangSmi...,To create user feedback with the LangSmith SDK...,None,To create user feedback with the LangSmith SDK...,1,0.492711,8401e038-1738-4410-9b1a-89dc755696ea,01a11cb2-8bd9-7463-8ce6-e6e823bbce4b
5,Is there a JavaScript LangSmith SDK?,"Yes, there is a JavaScript/TypeScript LangSmit...",None,"Yes, there is a JavaScript LangSmith SDK!",0,0.415699,eb396e7b-683b-4a6a-a085-86ed959ded80,01a11cb2-8dc7-7d82-9f74-d40b18308955
6,How do I set up tracing to LangSmith if I'm us...,LangSmith is natively integrated with LangChai...,None,Set your LangChain app to use LangSmith’s trac...,1,0.437224,c7fb22d1-c4c5-49dd-92c3-da90a804bc97,01a11cb2-8f67-7791-acb3-321b03fa56a4
7,How can I trace with the @traceable decorator?,You can trace a synchronous function by applyi...,None,To trace with the @traceable decorator in Pyth...,1,1.189760,0e3e0494-ccd1-4a20-b605-be94e398fee5,01a11cb2-911d-77c0-b6fe-96fbb2acc296
8,What is LangSmith used for in three sentences?,LangSmith is used for tracing LLM applications...,None,LangSmith is a platform designed for the devel...,1,0.449079,34bf2ff8-0797-4828-a770-cb6289e29049,01a11cb2-95c5-77f1-9b20-d5dcdcc31ab5
9,What testing capabilities does LangSmith have?,The provided context does not contain informat...,None,LangSmith offers capabilities for creating dat...,1,0.421076,604440fa-b554-46d5-aad2-2a5e41d3759d,01a11cb2-9787-7191-a278-2005b66eee35


### Running over Different pieces of Data

##### Dataset Version

You can execute an experiment on a specific version of a dataset in the sdk by using the `as_of` parameter in `list_examples`

Let's try running on just our initial dataset.

In [6]:
evaluate(
    target_function,
    data=client.list_examples(dataset_name=dataset_name, as_of="Initial Dataset"),   # We use as_of to specify a version
    evaluators=[is_concise_enough],
    experiment_prefix="initial dataset version"
)

View the evaluation results for experiment: 'initial dataset version-b9599e5a' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=add6c356-182c-4830-90b3-289f227508a5




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,Pass a dictionary of metadata to the `traceabl...,None,You can pass metadata with the @traceable deco...,1,1.882791,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a117a6-98e6-7131-9636-1bfc6ac45ac3
1,How can I trace with the @traceable decorator?,Add the @traceable,None,To trace with the @traceable decorator in Pyth...,1,0.754283,0e3e0494-ccd1-4a20-b605-be94e398fee5,01a117a6-a043-7252-8c47-43d8908fd6f3
2,Does LangSmith support offline evaluation?,Yes. LangSmith provides offline evaluation for...,None,"Yes, LangSmith supports offline evaluation thr...",1,0.643850,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a117a6-a336-7022-80cf-9636df335754
3,What is LangSmith used for in three sentences?,LangSmith is a platform for tracing and monito...,None,LangSmith is a platform designed for the devel...,1,0.719252,34bf2ff8-0797-4828-a770-cb6289e29049,01a117a6-a5bb-70b0-b427-88236790d8f8
4,Can LangSmith be used for finetuning and model...,"No, LangSmith is an observability and evaluati...",None,"Yes, LangSmith can be used for fine-tuning and...",1,0.694872,366693f8-2a71-43fc-9118-3d0427fb6c79,01a117a6-a88c-7801-8363-3409af04a645
5,What testing capabilities does LangSmith have?,LangSmith provides built‑in evaluation tools t...,None,LangSmith offers capabilities for creating dat...,1,0.699478,604440fa-b554-46d5-aad2-2a5e41d3759d,01a117a6-ab44-7393-bf55-57fc15e9cacd
6,Can LangSmith be used to evaluate agents?,Yes. LangSmith lets you define evaluators (via...,None,"Yes, LangSmith can be used to evaluate agents....",1,0.671302,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a117a6-ae00-7210-8c92-fc3a6ac2c48f
7,How do I set up tracing to LangSmith if I'm us...,To enable LangSmith tracing in,None,To set up tracing to LangSmith while using Lan...,1,1.986116,78a032f4-76d5-4d0d-a643-2def2eb72fd2,01a117a6-b0a1-74f2-ae93-bb6d52a669af
8,How do I create user feedback with the LangSmi...,You can log feedback directly with the LangSmi...,None,To create user feedback with the LangSmith SDK...,1,6.242818,8401e038-1738-4410-9b1a-89dc755696ea,01a117a6-b865-7fb1-8d09-b3f48d15ef90
9,Does LangSmith support online evaluation?,Yes. LangSmith provides online evaluators that...,None,"Yes, LangSmith supports online evaluation as a...",1,4.680021,b6f3accf-8938-4253-9cd6-49805712d50b,01a117a6-d0ca-7181-8396-25e62f3e5e74


##### Dataset Split

You can run an experiment on a specific split of your dataset, let's try running on the Crucial Examples split.

In [7]:
evaluate(
    target_function,
    data=client.list_examples(dataset_name=dataset_name, splits=["Crucial Examples"]),  # We pass in a list of Splits
    evaluators=[is_concise_enough],
    experiment_prefix="Crucial Examples split"
)

View the evaluation results for experiment: 'Crucial Examples split-24c329c5' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=19d797b1-636f-48d3-b56c-87909dcb89eb




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,Pass,None,You can pass metadata with the @traceable deco...,1,0.905930,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a117a9-5d7f-7de1-9cd9-49c694c585b6
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation. It...",None,"Yes, LangSmith supports offline evaluation thr...",1,0.768746,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a117a9-610b-7192-a783-66571f95c825
2,Can LangSmith be used for finetuning and model...,LangSmith is an observability and evaluation p...,None,"Yes, LangSmith can be used for fine-tuning and...",1,0.898441,366693f8-2a71-43fc-9118-3d0427fb6c79,01a117a9-640c-7852-9dbe-df6220182a4c
3,Can LangSmith be used to evaluate agents?,Yes. LangSmith’s evaluation features (via the ...,None,"Yes, LangSmith can be used to evaluate agents....",1,0.789537,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a117a9-6790-7123-9fe0-68c5edcb08ba
4,How do I create user feedback with the LangSmi...,Use the LangSmith SDK’s feedback client. For e...,None,To create user feedback with the LangSmith SDK...,1,0.945519,8401e038-1738-4410-9b1a-89dc755696ea,01a117a9-6aa7-72a1-8c25-034c8d257d94
5,Is there a JavaScript LangSmith SDK?,Yes—LangSmith provides a JavaScript/TypeScript...,None,"Yes, there is a JavaScript LangSmith SDK!",0,0.565924,eb396e7b-683b-4a6a-a085-86ed959ded80,01a117a9-6e5a-7993-b27d-c6ef826f32f1


##### Specific Data Points

You can specify individual data points to run an experiment over as well

In [8]:
# Grab the first 2 example IDs dynamically
sample_examples = list(client.list_examples(dataset_name=dataset_name, limit=2))
two_example_ids = [str(ex.id) for ex in sample_examples]

print(f"Testing specifically on IDs: {two_example_ids}")

evaluate(
    target_function,
    data=client.list_examples(
        dataset_name=dataset_name, 
        example_ids=two_example_ids
    ),
    evaluators=[is_concise_enough],
    experiment_prefix="two specific example ids"
)

Testing specifically on IDs: ['0e04b603-0839-4a85-846c-1b9ea5f18cb8', '1f8b6be5-223e-4ef2-8888-dd7ede3c5c16']
View the evaluation results for experiment: 'two specific example ids-0c651f79' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=24d68cab-247b-464a-b236-565d69c1254f




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,,None,You can pass metadata with the @traceable deco...,0,1.129159,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a117ab-4989-7142-aff4-a007f3ecc86e
1,Does LangSmith support offline evaluation?,Yes—LangSmith includes offline evaluation capa...,None,"Yes, LangSmith supports offline evaluation thr...",1,0.548266,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a117ab-4df4-7161-a538-42ba8baaf1a0


### Other Parameters

##### Repetitions

You can run an experiment several times to make sure you have consistent results

In [ ]:
# Repetitions
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="two repetitions",
    num_repetitions=2
)

View the evaluation results for experiment: 'concurrency-gpt-20b-5c08d417' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=755f9f06-9d4c-4c53-8cb4-8d3d067c68f8




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,Does LangSmith support offline evaluation?,Yes. LangSmith includes **offline evaluation**...,None,"Yes, LangSmith supports offline evaluation thr...",1,0.797602,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a117b2-7424-7520-86da-4552fb341ef6
1,How do I pass metadata in with @traceable?,,None,You can pass metadata with the @traceable deco...,0,0.851757,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a117b2-7423-7f62-a7dd-d34d7978ac34
2,Can LangSmith be used for finetuning and model...,LangSmith is a monitoring and evaluation platf...,None,"Yes, LangSmith can be used for fine-tuning and...",1,0.515536,366693f8-2a71-43fc-9118-3d0427fb6c79,01a117b2-7742-72d1-82eb-c8c98e37d514
3,Can LangSmith be used to evaluate agents?,,None,"Yes, LangSmith can be used to evaluate agents....",0,0.675442,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a117b2-7777-7a62-af5c-7cb5e797d2de
4,How do I create user feedback with the LangSmi...,Use the,None,To create user feedback with the LangSmith SDK...,1,0.493522,8401e038-1738-4410-9b1a-89dc755696ea,01a117b2-7946-79e1-8f36-9eeccbec31b8
5,Is there a JavaScript LangSmith SDK?,"Yes, LangSmith offers a JavaScript/TypeScript ...",None,"Yes, there is a JavaScript LangSmith SDK!",0,0.424461,eb396e7b-683b-4a6a-a085-86ed959ded80,01a117b2-7a1b-7282-90b3-063bb416f293
6,How do I set up tracing to LangSmith if I'm us...,,None,Set your LangChain app to use LangSmith’s trac...,0,0.574188,c7fb22d1-c4c5-49dd-92c3-da90a804bc97,01a117b2-7b34-7f50-a8f7-e70cc51b782f
7,How can I trace with the @traceable decorator?,Use the LangSmith **@traceable** decorator by ...,None,To trace with the @traceable decorator in Pyth...,1,0.681060,0e3e0494-ccd1-4a20-b605-be94e398fee5,01a117b2-7bc3-7752-8748-af5397af8512
8,What is LangSmith used for in three sentences?,LangSmith is a tracing and monitoring platform...,None,LangSmith is a platform designed for the devel...,1,0.463429,34bf2ff8-0797-4828-a770-cb6289e29049,01a117b2-7d73-7563-8783-3b04691e65d4
9,What testing capabilities does LangSmith have?,LangSmith supports a full testing workflow: yo...,None,LangSmith offers capabilities for creating dat...,1,0.466412,604440fa-b554-46d5-aad2-2a5e41d3759d,01a117b2-7e6d-7f00-bb20-f6f71d4533f7


##### Concurrency
You can also kick off concurrent threads of execution to make your experiments finish faster!

In [12]:
evaluate(
    lambda inputs: langsmith_rag(inputs["question"], model="openai/gpt-oss-20b"),
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="concurrency-gpt-20b",
    max_concurrency=2
)

View the evaluation results for experiment: 'concurrency-gpt-20b-9c01431f' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=4562672f-3728-416c-83ca-9a3ae96eae9e




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,,None,You can pass metadata with the @traceable deco...,0,0.894576,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a117b5-921f-70f0-91e4-419f99d932e4
1,Does LangSmith support offline evaluation?,Yes. LangSmith includes an offline evaluation ...,None,"Yes, LangSmith supports offline evaluation thr...",1,0.908843,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a117b5-9220-7910-a145-8d5dff68c901
2,Can LangSmith be used for finetuning and model...,,None,"Yes, LangSmith can be used for fine-tuning and...",0,0.604973,366693f8-2a71-43fc-9118-3d0427fb6c79,01a117b5-959e-7e51-a46b-690b0ea63a67
3,Can LangSmith be used to evaluate agents?,Yes—LangSmith can evaluate agents. You can cre...,None,"Yes, LangSmith can be used to evaluate agents....",1,0.621721,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a117b5-95af-7720-b080-1c271157f2fe
4,How do I create user feedback with the LangSmi...,,None,To create user feedback with the LangSmith SDK...,0,0.556832,8401e038-1738-4410-9b1a-89dc755696ea,01a117b5-97fb-7fd0-9518-265879778310
5,Is there a JavaScript LangSmith SDK?,"Yes, LangSmith provides a JavaScript/TypeScrip...",None,"Yes, there is a JavaScript LangSmith SDK!",0,0.651635,eb396e7b-683b-4a6a-a085-86ed959ded80,01a117b5-981d-76d2-82df-1bdf19a7f9fe
6,How do I set up tracing to LangSmith if I'm us...,,None,Set your LangChain app to use LangSmith’s trac...,0,0.642518,c7fb22d1-c4c5-49dd-92c3-da90a804bc97,01a117b5-9a29-7d62-bc96-ba8c674c89ef
7,How can I trace with the @traceable decorator?,"To trace a function with LangSmith, decorate i...",None,To trace with the @traceable decorator in Pyth...,1,0.958614,0e3e0494-ccd1-4a20-b605-be94e398fee5,01a117b5-9aaa-7982-ba10-d044fe545447
8,What is LangSmith used for in three sentences?,,None,LangSmith is a platform designed for the devel...,0,0.558267,34bf2ff8-0797-4828-a770-cb6289e29049,01a117b5-9cac-72c3-8eb4-fbedfc96a09f
9,How do I set up tracing to LangSmith if I'm us...,,None,To set up tracing to LangSmith while using Lan...,0,0.559481,78a032f4-76d5-4d0d-a643-2def2eb72fd2,01a117b5-9edb-7d63-b4d1-0991588f93be


##### Metadata 

You can (and should) add metadata to your experiments, to make them easier to find in the UI

In [13]:

# Metadata
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="metadata added",
    metadata={
        "model_name": "openai/gpt-oss-120b",
        "provider": "groq",
        "env": "testing"
    }
)

View the evaluation results for experiment: 'metadata added-c862555a' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=5bcfa45c-9c57-44d2-b490-249a66a7571e




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,Pass a `metadata`,None,You can pass metadata with the @traceable deco...,1,0.884179,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a117b6-c747-7d92-b1fa-651875428447
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation. It...",None,"Yes, LangSmith supports offline evaluation thr...",1,0.547655,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a117b6-cabd-70d0-866a-e50af1b66c2d
2,Can LangSmith be used for finetuning and model...,No; LangSmith is a monitoring and observabilit...,None,"Yes, LangSmith can be used for fine-tuning and...",1,0.644606,366693f8-2a71-43fc-9118-3d0427fb6c79,01a117b6-cce1-72a1-a8d4-36da2426c9c2
3,Can LangSmith be used to evaluate agents?,Yes. LangSmith’s evaluation framework works wi...,None,"Yes, LangSmith can be used to evaluate agents....",1,0.892497,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a117b6-cf67-79f3-899c-63fade7044c8
4,How do I create user feedback with the LangSmi...,"To log feedback with the LangSmith SDK, first ...",None,To create user feedback with the LangSmith SDK...,1,0.747715,8401e038-1738-4410-9b1a-89dc755696ea,01a117b6-d2e5-7843-9892-b5594a4eaaf1
5,Is there a JavaScript LangSmith SDK?,Yes—LangSmith provides a JavaScript/TypeScript...,None,"Yes, there is a JavaScript LangSmith SDK!",0,0.652707,eb396e7b-683b-4a6a-a085-86ed959ded80,01a117b6-d5d1-7820-837e-4f9f8a832e7c
6,How do I set up tracing to LangSmith if I'm us...,"To enable LangSmith tracing with LangChain, fi...",None,Set your LangChain app to use LangSmith’s trac...,1,0.801521,c7fb22d1-c4c5-49dd-92c3-da90a804bc97,01a117b6-d860-7853-a8fb-db0653bdd0d9
7,How can I trace with the @traceable decorator?,Apply the @traceable decorator to the function...,None,To trace with the @traceable decorator in Pyth...,1,0.693761,0e3e0494-ccd1-4a20-b605-be94e398fee5,01a117b6-db83-7a43-bbf7-2f7681aa5eec
8,What is LangSmith used for in three sentences?,LangSmith is a monitoring and observability pl...,None,LangSmith is a platform designed for the devel...,1,2.013587,34bf2ff8-0797-4828-a770-cb6289e29049,01a117b6-de39-7d81-9c03-a73468ed8dfc
9,What testing capabilities does LangSmith have?,,None,LangSmith offers capabilities for creating dat...,0,0.700594,604440fa-b554-46d5-aad2-2a5e41d3759d,01a117b6-e618-7601-99f5-c05834159c93


In [6]:
evaluate(
    target_function_qwen,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="groq-qwen-27b",
    metadata={
        "model_name": "qwen/qwen3.8-27b",
        "provider": "groq",
        "env": "testing"
    }
)

View the evaluation results for experiment: 'groq-qwen-27b-b128954c' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=b7ddecdc-244e-490f-9a6a-d54a4c3445a6




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,You cannot pass metadata directly as an argume...,None,You can pass metadata with the @traceable deco...,1,0.509665,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a11cb2-be8c-7a61-b604-bb60e0a9d8f8
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluations, w...",None,"Yes, LangSmith supports offline evaluation thr...",1,0.367708,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a11cb2-c08d-70d0-9d89-92147bf2b3c9
2,Can LangSmith be used for finetuning and model...,"No, the provided context indicates that LangSm...",None,"Yes, LangSmith can be used for fine-tuning and...",1,2.339779,366693f8-2a71-43fc-9118-3d0427fb6c79,01a11cb2-c1fd-7ce1-810f-93769c0ac1a4
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents,...",None,"Yes, LangSmith can be used to evaluate agents....",1,3.773794,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a11cb2-cb22-7522-b080-2beac108e63e
4,How do I create user feedback with the LangSmi...,"In LangSmith, create user feedback using the `...",None,To create user feedback with the LangSmith SDK...,1,4.479020,8401e038-1738-4410-9b1a-89dc755696ea,01a11cb2-d9e2-7eb2-9c3b-defee6ecfff1
5,Is there a JavaScript LangSmith SDK?,"Yes, there is a JavaScript LangSmith SDK, whic...",None,"Yes, there is a JavaScript LangSmith SDK!",0,5.645273,eb396e7b-683b-4a6a-a085-86ed959ded80,01a11cb2-eb62-7592-98d6-020af6468043
6,How do I set up tracing to LangSmith if I'm us...,The provided retrieved context focuses on Lang...,None,Set your LangChain app to use LangSmith’s trac...,1,4.514988,c7fb22d1-c4c5-49dd-92c3-da90a804bc97,01a11cb3-0171-7263-a7bf-c58482a4554c
7,How can I trace with the @traceable decorator?,Apply the `@traceable` decorator to any Python...,None,To trace with the @traceable decorator in Pyth...,1,8.570314,0e3e0494-ccd1-4a20-b605-be94e398fee5,01a11cb3-1315-7163-95d3-82139aed98bd
8,What is LangSmith used for in three sentences?,LangSmith is an LLM engineering platform used ...,None,LangSmith is a platform designed for the devel...,1,2.545899,34bf2ff8-0797-4828-a770-cb6289e29049,01a11cb3-3490-7d11-ada6-ec213306e390
9,What testing capabilities does LangSmith have?,"Based on the provided context, there is no men...",None,LangSmith offers capabilities for creating dat...,1,4.562936,604440fa-b554-46d5-aad2-2a5e41d3759d,01a11cb3-3e85-7412-b911-4d0d9fd835c3


In [7]:
evaluate(
    lambda inputs: langsmith_rag(inputs["question"], model="openai/gpt-oss-20b"),
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="NOTconcurrency-gpt-20b",

)

View the evaluation results for experiment: 'NOTconcurrency-gpt-20b-f5775b3a' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/cb14c20c-b45e-45a1-9701-5558e8a8a945/compare?selectedSessions=c510a6df-0a6c-4bea-8df7-40f5a386ac8e




,inputs.question,outputs.output,error,reference.output,feedback.is_concise,execution_time,example_id,id
0,How do I pass metadata in with @traceable?,In LangSmith you can attach metadata to a run ...,None,You can pass metadata with the @traceable deco...,1,0.827069,0e04b603-0839-4a85-846c-1b9ea5f18cb8,01a11cd3-12b5-7ba1-a452-724019f77d99
1,Does LangSmith support offline evaluation?,"Yes, LangSmith supports offline evaluation. \...",None,"Yes, LangSmith supports offline evaluation thr...",1,0.553309,1f8b6be5-223e-4ef2-8888-dd7ede3c5c16,01a11cd3-15f2-73b2-b228-aca964e9ecb2
2,Can LangSmith be used for finetuning and model...,No. LangSmith is an observability and monitori...,None,"Yes, LangSmith can be used for fine-tuning and...",1,0.468361,366693f8-2a71-43fc-9118-3d0427fb6c79,01a11cd3-181d-73a1-9e9b-ec7b3f038e52
3,Can LangSmith be used to evaluate agents?,Yes. LangSmith supports evaluating agents thro...,None,"Yes, LangSmith can be used to evaluate agents....",1,0.483419,61e3a897-7321-4d84-a0b8-9a07731ff50c,01a11cd3-19f2-74a3-865c-a884e959a57c
4,How do I create user feedback with the LangSmi...,,None,To create user feedback with the LangSmith SDK...,0,0.435471,8401e038-1738-4410-9b1a-89dc755696ea,01a11cd3-1bd8-7c71-b19c-55abcc546589
5,Is there a JavaScript LangSmith SDK?,Yes. LangSmith provides a JavaScript/TypeScrip...,None,"Yes, there is a JavaScript LangSmith SDK!",0,0.566082,eb396e7b-683b-4a6a-a085-86ed959ded80,01a11cd3-1d8c-7660-b65f-da00da7601f2
6,How do I set up tracing to LangSmith if I'm us...,"To trace a LangChain app with LangSmith, first...",None,Set your LangChain app to use LangSmith’s trac...,1,0.575216,c7fb22d1-c4c5-49dd-92c3-da90a804bc97,01a11cd3-1fc3-7792-bec8-cab302d6d5e9
7,How can I trace with the @traceable decorator?,Use the `@traceable` decorator from LangSmith ...,None,To trace with the @traceable decorator in Pyth...,1,0.515301,0e3e0494-ccd1-4a20-b605-be94e398fee5,01a11cd3-2203-7ee0-a69d-8c6661863fac
8,What is LangSmith used for in three sentences?,LangSmith is a platform for tracing and monito...,None,LangSmith is a platform designed for the devel...,1,0.443948,34bf2ff8-0797-4828-a770-cb6289e29049,01a11cd3-2408-7da2-859c-7501316bd3a4
9,What testing capabilities does LangSmith have?,,None,LangSmith offers capabilities for creating dat...,0,0.561842,604440fa-b554-46d5-aad2-2a5e41d3759d,01a11cd3-25c4-7933-8cda-0047c42667c3
